# Gold: patient_360

Builds `adb_medsync.gold.patient_360` per the Silver-to-Gold STTM: one row per patient, enriched with hospital and insurance provider names and lab test aggregates. Full rebuild (CREATE OR REPLACE TABLE AS SELECT); LEFT JOINs from `silver.patients` so no patient is dropped.

In [0]:
%sql
-- Target: adb_medsync.gold.patient_360 | Grain: one row per patient | Load: FULL
CREATE OR REPLACE TABLE adb_medsync.gold.patient_360
COMMENT 'Gold: one row per patient with hospital, insurance and lab test summary (STTM silver-to-gold)'
AS
WITH lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                                              AS total_tests,
    COUNT_IF(result_status = 'Abnormal')                  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')                  AS critical_test_count,
    MAX(test_date)                                        AS last_test_date
  FROM adb_medsync.silver.lab_results
  GROUP BY patient_id
)
SELECT
  p.patient_id                                                         AS patient_id,
  p.first_name                                                         AS first_name,
  p.last_name                                                          AS last_name,
  CAST(p.date_of_birth AS DATE)                                        AS date_of_birth,
  CAST(FLOOR(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender                                                             AS gender,
  p.city                                                               AS city,
  p.state                                                              AS state,
  p.registered_hospital_id                                             AS hospital_id,
  h.hospital_name                                                      AS hospital_name,
  p.insurance_provider_id                                              AS insurance_provider_id,
  i.provider_name                                                      AS insurance_provider_name,
  CAST(COALESCE(l.total_tests, 0) AS INT)                              AS total_tests,
  CAST(COALESCE(l.abnormal_test_count, 0) AS INT)                      AS abnormal_test_count,
  CAST(COALESCE(l.critical_test_count, 0) AS INT)                      AS critical_test_count,
  l.last_test_date                                                     AS last_test_date
FROM adb_medsync.silver.patients p
LEFT JOIN adb_medsync.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN adb_medsync.silver.insurance_providers i
  ON p.insurance_provider_id = i.insurance_provider_id
LEFT JOIN lab_agg l
  ON p.patient_id = l.patient_id

In [0]:
target_table = "adb_medsync.gold.patient_360"
updated_df = spark.read.table(target_table)
record_count = updated_df.count()

print(f"Records loaded into {target_table}: {record_count}")
dbutils.notebook.exit(str(record_count))